In [2]:
import numpy as np
import tqdm
import os
import json
import time
import logging
import traceback
import glob
from datetime import datetime
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# ======================================================================
# 1. Base graph coevolutionary algorithm
# ======================================================================

class Graph_PDCoEA:
    """Pairwise-dominance coevolutionary algorithm played on a graph."""

    def __init__(self, alpha, beta, chi, l, n, t, edge_set, num_nodes, timestep=10, probs=None):
        probs = None if probs is None else np.asarray(probs)
        if probs is None:
            self.p_fixed = np.random.rand(num_nodes)
            self.q_fixed = np.random.rand(num_nodes)
        elif probs.ndim == 1:
            self.p_fixed = probs
            self.q_fixed = probs
        elif probs.ndim == 2 and probs.shape[1] == 2:
            self.p_fixed = probs[:, 0]
            self.q_fixed = probs[:, 1]

        self.alpha, self.beta, self.chi = alpha, beta, chi
        self.l, self.n, self.t = l, n, t
        self.num_nodes = num_nodes
        self.edge_set = np.asarray(edge_set)
        self.timestep = timestep

        self.P = None
        self.Q = None
        self.snapshots_P = None
        self.snapshots_Q = None
        self._snapshot_gens = None

    def fitness(self, x, y):
        return (((x - self.n / 2) ** 3 - 3 * (self.alpha * self.n / 2) ** 2 * (x - self.n / 2)) +
                ((y - self.n / 2) ** 3 - 3 * (self.alpha * self.n / 2) ** 2 * (y - self.n / 2)))

    def _apply_selection(self, select, edge, x_1, y_1, x_2, y_2):
        self.P[edge[0]] = np.where(select[:, None], x_1, x_2)
        self.Q[edge[1]] = np.where(select[:, None], y_1, y_2)
        self.P[edge[0]] ^= (np.random.random((self.l, self.n)) <= self.chi / self.n)
        self.Q[edge[1]] ^= (np.random.random((self.l, self.n)) <= self.chi / self.n)

    def selection(self, P_1, P_2, edge):
        x_1, y_1 = P_1
        x_2, y_2 = P_2

        c1 = self.fitness(x_1.sum(axis=1), y_2.sum(axis=1)) >= self.fitness(x_1.sum(axis=1), y_1.sum(axis=1))
        c2 = self.fitness(x_1.sum(axis=1), y_1.sum(axis=1)) >= self.fitness(x_2.sum(axis=1), y_1.sum(axis=1))
        self._apply_selection(np.logical_and(c1, c2), edge, x_1, y_1, x_2, y_2)

    def initialize_population(self):
        p = self.p_fixed.reshape(self.num_nodes, 1, 1)
        q = self.q_fixed.reshape(self.num_nodes, 1, 1)
        self.P = (np.random.rand(self.num_nodes, self.l, self.n) < p).astype(np.int8)
        self.Q = (np.random.rand(self.num_nodes, self.l, self.n) < q).astype(np.int8)

    def sample_from_pop(self):
        edge = self.edge_set[np.random.choice(self.edge_set.shape[0])]
        P_1 = (self.P[edge[0]][np.random.choice(self.l, size=self.l)],
               self.Q[edge[1]][np.random.choice(self.l, size=self.l)])
        P_2 = (self.P[edge[0]][np.random.choice(self.l, size=self.l)],
               self.Q[edge[1]][np.random.choice(self.l, size=self.l)])
        return P_1, P_2, edge

    def record_snapshot(self, gen=None):
        """Copy current P, Q per node as int8."""
        if self.snapshots_P is None:
            self.snapshots_P = {node: [] for node in range(self.num_nodes)}
            self.snapshots_Q = {node: [] for node in range(self.num_nodes)}
            self._snapshot_gens = []

        for node in range(self.num_nodes):
            self.snapshots_P[node].append(self.P[node].astype(np.int8))
            self.snapshots_Q[node].append(self.Q[node].astype(np.int8))
        self._snapshot_gens.append(gen)

    def num_snapshots(self):
        return 0 if self.snapshots_P is None else len(self.snapshots_P[0])

    def run(self, track_champions=False):
        """Runs the algorithm, optionally recording snapshots and champions."""
        self.initialize_population()
        for t in tqdm.tqdm(range(self.t)):
            P_1, P_2, edge = self.sample_from_pop()
            self.selection(P_1, P_2, edge)

            if t % self.timestep == 0:
                self.record_snapshot(gen=t)
                if track_champions:
                    self.record_champions()

    def _save_params(self, save_dir):
        np.savez(os.path.join(save_dir, "params.npz"),
                 alpha=self.alpha, beta=self.beta, chi=self.chi, l=self.l, n=self.n, t=self.t,
                 num_nodes=self.num_nodes, timestep=self.timestep, edge_set=self.edge_set)

    def save_results(self, save_dir):
        os.makedirs(save_dir, exist_ok=True)
        for node in range(self.num_nodes):
            node_dir = os.path.join(save_dir, f"node_{node}")
            os.makedirs(node_dir, exist_ok=True)
            if self.snapshots_P is not None:
                np.savez(os.path.join(node_dir, "snapshots.npz"),
                         P=np.array(self.snapshots_P[node], dtype=np.int8),
                         Q=np.array(self.snapshots_Q[node], dtype=np.int8))

        if self._snapshot_gens is not None:
            np.savez(os.path.join(save_dir, "snapshot_gens.npz"), gens=np.array(self._snapshot_gens))
        self._save_params(save_dir)

    def load_snapshots(self, save_dir):
        self.snapshots_P, self.snapshots_Q = {}, {}
        for node in range(self.num_nodes):
            data = np.load(os.path.join(save_dir, f"node_{node}", "snapshots.npz"))
            self.snapshots_P[node] = list(data["P"])
            self.snapshots_Q[node] = list(data["Q"])
        gens_path = os.path.join(save_dir, "snapshot_gens.npz")
        if os.path.exists(gens_path):
            self._snapshot_gens = list(np.load(gens_path)["gens"])


# ======================================================================
# 2. DefendIt game + graph algorithm
# ======================================================================

class DefendItGame:
    """Pure DefendIt payoff logic (resource ownership, costs, payoffs)."""

    def __init__(self, k, l_t, v, c, B_D, B_A):
        self.k, self.l_t = k, l_t
        self.v, self.c = np.asarray(v), np.asarray(c)
        self.B_D, self.B_A = B_D, B_A

    def ownership(self, x, y):
        x_r = x.reshape(x.shape[0], self.k, self.l_t)
        y_r = y.reshape(y.shape[0], self.k, self.l_t)
        z = np.ones((x.shape[0], self.k, self.l_t + 1), dtype=np.int8)
        for i in range(self.l_t):
            same = x_r[:, :, i] == y_r[:, :, i]
            z[:, :, i + 1] = np.where(same, z[:, :, i], x_r[:, :, i])
        return z[:, :, 1:]

    def cost(self, x):
        return (x.reshape(x.shape[0], self.k, self.l_t).sum(axis=2) * self.c).sum(axis=1)

    def payoff_defender(self, x, y):
        z = self.ownership(x, y)
        within_budget = self.cost(x) <= self.B_D
        return np.where(within_budget, (z.sum(axis=2) * self.v).sum(axis=1), -x.sum(axis=1))

    def payoff_attacker(self, x, y):
        z = self.ownership(x, y)
        within_budget = self.cost(y) <= self.B_A
        return np.where(within_budget, ((self.l_t - z.sum(axis=2)) * self.v).sum(axis=1), -y.sum(axis=1))

    def payoff_matrix_defender(self, X, Y):
        X_rep, Y_tile = np.repeat(X, Y.shape[0], axis=0), np.tile(Y, (X.shape[0], 1))
        return self.payoff_defender(X_rep, Y_tile).reshape(X.shape[0], Y.shape[0])

    def payoff_matrix_attacker(self, X, Y):
        X_tile, Y_rep = np.tile(X, (Y.shape[0], 1)), np.repeat(Y, X.shape[0], axis=0)
        return self.payoff_attacker(X_tile, Y_rep).reshape(Y.shape[0], X.shape[0]).T


class Graph_PDCoEA_Defendit(Graph_PDCoEA, DefendItGame):
    """Graph-topology PDCoEA played on the DefendIt game."""

    def __init__(self, alpha, beta, chi, l, n, t, edge_set, num_nodes, k, l_t, v, c, B_D, B_A,
                 timestep=10, probs=None, topology_name=None):
        Graph_PDCoEA.__init__(self, alpha, beta, chi, l, n, t, edge_set, num_nodes, timestep=timestep, probs=probs)
        DefendItGame.__init__(self, k, l_t, v, c, B_D, B_A)
        if k * l_t != n:
            raise ValueError("k*l_t must equal n")

        self.topology_name = topology_name
        self.defender_champions = None
        self.attacker_champions = None

    def selection(self, P_1, P_2, edge):
        x_1, y_1 = P_1
        x_2, y_2 = P_2

        select = np.logical_and(
            self.payoff_defender(x_1, y_1) >= self.payoff_defender(x_2, y_1),
            self.payoff_attacker(x_1, y_1) >= self.payoff_attacker(x_1, y_2),
        )
        self._apply_selection(select, edge, x_1, y_1, x_2, y_2)

    def defender_population(self):
        return self.P.reshape(-1, self.n)

    def attacker_population(self):
        return self.Q.reshape(-1, self.n)

    def neighbors(self, node):
        mask_out, mask_in = self.edge_set[:, 0] == node, self.edge_set[:, 1] == node
        return np.unique(np.concatenate([self.edge_set[mask_out, 1], self.edge_set[mask_in, 0]]))

    def _get_champion(self, node, is_defender=True):
        if is_defender:
            opp_pool = np.concatenate([self.Q[v] for v in self.neighbors(node)], axis=0)
            worst_case = self.payoff_matrix_defender(self.P[node], opp_pool).min(axis=1)
            return self.P[node][np.argmax(worst_case)]
        else:
            opp_pool = np.concatenate([self.P[v] for v in self.neighbors(node)], axis=0)
            worst_case = self.payoff_matrix_attacker(opp_pool, self.Q[node]).min(axis=0)
            return self.Q[node][np.argmax(worst_case)]

    def record_champions(self):
        if self.defender_champions is None:
            self.defender_champions = {node: [] for node in range(self.num_nodes)}
            self.attacker_champions = {node: [] for node in range(self.num_nodes)}
        for node in range(self.num_nodes):
            self.defender_champions[node].append(self._get_champion(node, is_defender=True))
            self.attacker_champions[node].append(self._get_champion(node, is_defender=False))

    def load_snapshot_state(self, index):
        self.P = np.array([self.snapshots_P[node][index] for node in range(self.num_nodes)])
        self.Q = np.array([self.snapshots_Q[node][index] for node in range(self.num_nodes)])

    def champions_at_snapshot(self, index):
        self.load_snapshot_state(index)
        return ({node: self._get_champion(node, is_defender=True) for node in range(self.num_nodes)},
                {node: self._get_champion(node, is_defender=False) for node in range(self.num_nodes)})

    def compute_all_champions(self):
        """Computes champions offline from recorded/loaded snapshots."""
        final_P, final_Q = self.P, self.Q
        self.defender_champions = {node: [] for node in range(self.num_nodes)}
        self.attacker_champions = {node: [] for node in range(self.num_nodes)}
        for i in range(self.num_snapshots()):
            defenders, attackers = self.champions_at_snapshot(i)
            for node in range(self.num_nodes):
                self.defender_champions[node].append(defenders[node])
                self.attacker_champions[node].append(attackers[node])
        self.P, self.Q = final_P, final_Q

    def _save_params(self, save_dir):
        np.savez(os.path.join(save_dir, "params.npz"),
                 alpha=self.alpha, beta=self.beta, chi=self.chi, l=self.l, n=self.n, t=self.t,
                 num_nodes=self.num_nodes, timestep=self.timestep, k=self.k, l_t=self.l_t,
                 v=self.v, c=self.c, B_D=self.B_D, B_A=self.B_A, edge_set=self.edge_set)

    def save_results(self, save_dir):
        super().save_results(save_dir)

        if self.defender_champions is not None:
            for node in range(self.num_nodes):
                np.savez(os.path.join(save_dir, f"node_{node}", "champions.npz"),
                         defender=np.array(self.defender_champions[node]),
                         attacker=np.array(self.attacker_champions[node]))

        meta = {
            "topology_name": self.topology_name, "num_nodes": self.num_nodes,
            "edge_set": self.edge_set.tolist(), "k": self.k, "l_t": self.l_t,
            "n": self.n, "l": self.l, "t": self.t, "chi": self.chi, "timestep": self.timestep,
            "B_D": self.B_D, "B_A": self.B_A, "num_snapshots": self.num_snapshots(),
            "has_champions": self.defender_champions is not None,
        }
        with open(os.path.join(save_dir, "meta.json"), "w") as f:
            json.dump(meta, f, indent=2)


# ======================================================================
# 3. Overnight job queue
# ======================================================================

class ExperimentQueue:
    """Runs a resumable queue of experiments with per-job failure isolation."""

    def __init__(self, base_dir="experiments", resume=True):
        self.base_dir = base_dir
        self.resume = resume
        os.makedirs(base_dir, exist_ok=True)
        self.logger = self._make_logger()

    def _make_logger(self):
        logger = logging.getLogger(f"experiment_queue_{id(self)}")
        logger.setLevel(logging.INFO)
        logger.handlers = [
            logging.FileHandler(os.path.join(self.base_dir, "queue.log")),
            logging.StreamHandler()
        ]
        for handler in logger.handlers:
            handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
        return logger

    @staticmethod
    def _done_marker(directory):
        return os.path.join(directory, "_DONE")

    def _run_job(self, build_algo, n_repeats, save_dir, compute_champions=False):
        os.makedirs(save_dir, exist_ok=True)
        for i in range(n_repeats):
            run_dir = os.path.join(save_dir, f"run_{i}")
            if self.resume and os.path.exists(self._done_marker(run_dir)):
                self.logger.info(f"  run_{i}: already complete, skipping")
                continue

            self.logger.info(f"  run_{i}: starting")
            t0 = time.time()

            algo = build_algo()
            algo.run(track_champions=False)
            if compute_champions:
                algo.compute_all_champions()
            algo.save_results(save_dir=run_dir)

            with open(self._done_marker(run_dir), "w") as f:
                f.write(datetime.now().isoformat())
            self.logger.info(f"  run_{i}: done in {time.time() - t0:.1f}s")

        with open(self._done_marker(save_dir), "w") as f:
            f.write(datetime.now().isoformat())

    def run(self, jobs):
        self.logger.info(f"Starting queue with {len(jobs)} job(s): {[j['name'] for j in jobs]}")
        status = {}
        for job in jobs:
            name = job["name"]
            job_dir = os.path.join(self.base_dir, name)

            if self.resume and os.path.exists(self._done_marker(job_dir)):
                self.logger.info(f"[{name}] already complete, skipping")
                status[name] = "skipped (already complete)"
                continue

            self.logger.info(f"[{name}] starting")
            t0 = time.time()
            try:
                self._run_job(job["build_algo"], job["n_repeats"], job_dir,
                               compute_champions=job.get("compute_champions", False))
                status[name] = f"completed in {(time.time() - t0) / 60:.1f} min"
                self.logger.info(f"[{name}] {status[name]}")
            except Exception as e:
                status[name] = f"failed: {e}"
                self.logger.error(f"[{name}] FAILED: {e}")
                self.logger.error(traceback.format_exc())

            with open(os.path.join(self.base_dir, "queue_status.json"), "w") as f:
                json.dump(status, f, indent=2)

        self.logger.info("Queue finished.")
        return status


# ======================================================================
# 4. Offline comparison + plotting
# ======================================================================

class SnapshotComparison:
    """Analyzes saved snapshots for two topologies."""

    def __init__(self, root_dir, topology_A, topology_B):
        self.root_dir, self.topology_A, self.topology_B = root_dir, topology_A, topology_B

        params_A, params_B = self.load_params(topology_A, 0), self.load_params(topology_B, 0)
        for key in ("k", "l_t", "timestep"):
            if int(params_A[key]) != int(params_B[key]):
                raise ValueError(f"topology_A and topology_B must use the same {key}")
        for key in ("v", "c"):
            if not np.array_equal(params_A[key], params_B[key]):
                raise ValueError(f"topology_A and topology_B must use the same {key}")
        for key in ("B_D", "B_A"):
            if float(params_A[key]) != float(params_B[key]):
                raise ValueError(f"topology_A and topology_B must use the same {key}")

        self.period = int(params_A["timestep"])
        self.game = DefendItGame(k=int(params_A["k"]), l_t=int(params_A["l_t"]),
                                  v=params_A["v"], c=params_A["c"],
                                  B_D=float(params_A["B_D"]), B_A=float(params_A["B_A"]))

    def run_dir(self, topology, run_idx):
        return os.path.join(self.root_dir, topology, f"run_{run_idx}")

    def load_params(self, topology, run_idx):
        return np.load(os.path.join(self.run_dir(topology, run_idx), "params.npz"), allow_pickle=True)

    def load_snapshots(self, topology, run_idx):
        node_dirs = sorted(glob.glob(os.path.join(self.run_dir(topology, run_idx), "node_*")))
        if not node_dirs:
            raise ValueError(f"no node snapshots found for {topology} run {run_idx}")

        P_list, Q_list = [], []
        for node_dir in node_dirs:
            data = np.load(os.path.join(node_dir, "snapshots.npz"))
            P_list.append(data["P"])
            Q_list.append(data["Q"])
        return np.concatenate(P_list, axis=1), np.concatenate(Q_list, axis=1)

    def compare_run(self, run_idx):
        P_A, Q_A = self.load_snapshots(self.topology_A, run_idx)
        P_B, Q_B = self.load_snapshots(self.topology_B, run_idx)
        n_periods = min(P_A.shape[0], P_B.shape[0])

        defender_champions_A, defender_champions_B = [], []
        attacker_champions_A, attacker_champions_B = [], []
        performance = {k: [] for k in
                       ("performance_defender_A", "performance_defender_B",
                        "performance_attacker_A", "performance_attacker_B")}

        for i in range(n_periods):
            X_A, Y_A, X_B, Y_B = P_A[i], Q_A[i], P_B[i], Q_B[i]

            defender_champions_A.append(X_A[np.argmax(self.game.payoff_matrix_defender(X_A, Y_B).min(axis=1))])
            defender_champions_B.append(X_B[np.argmax(self.game.payoff_matrix_defender(X_B, Y_A).min(axis=1))])
            attacker_champions_A.append(Y_A[np.argmax(self.game.payoff_matrix_attacker(X_B, Y_A).min(axis=0))])
            attacker_champions_B.append(Y_B[np.argmax(self.game.payoff_matrix_attacker(X_A, Y_B).min(axis=0))])

            V_pool = np.array(attacker_champions_A + attacker_champions_B)
            U_pool = np.array(defender_champions_A + defender_champions_B)

            performance["performance_defender_A"].append(self.game.payoff_matrix_defender(X_A, V_pool).min(axis=1).max())
            performance["performance_defender_B"].append(self.game.payoff_matrix_defender(X_B, V_pool).min(axis=1).max())
            performance["performance_attacker_A"].append(self.game.payoff_matrix_attacker(U_pool, Y_A).min(axis=0).max())
            performance["performance_attacker_B"].append(self.game.payoff_matrix_attacker(U_pool, Y_B).min(axis=0).max())

        return {k: np.array(v) for k, v in performance.items()}

    def compare_all_runs(self, n_runs):
        results = [self.compare_run(i) for i in tqdm.tqdm(range(n_runs))]
        return {key: np.stack([r[key] for r in results]) for key in results[0]}

    def plot_whisker(self, n_runs, metric="performance_defender", save_dir="whisker_plots",
                      filename="whisker.html", label_A=None, label_B=None):
        results = self.compare_all_runs(n_runs)
        A, B = results[f"{metric}_A"], results[f"{metric}_B"]
        periods = np.arange(A.shape[1]) * self.period
        label_A, label_B = label_A or self.topology_A, label_B or self.topology_B

        os.makedirs(save_dir, exist_ok=True)
        fig = go.Figure()
        for i in range(A.shape[1]):
            fig.add_trace(go.Box(y=A[:, i], x=[periods[i]] * A.shape[0], name=label_A,
                                  marker_color="red", legendgroup=label_A, showlegend=(i == 0), offsetgroup=label_A))
            fig.add_trace(go.Box(y=B[:, i], x=[periods[i]] * B.shape[0], name=label_B,
                                  marker_color="blue", legendgroup=label_B, showlegend=(i == 0), offsetgroup=label_B))

        fig.update_layout(title=f"{metric} over time", xaxis_title="Time (f_evals)",
                           yaxis_title=metric, boxmode="group")
        fig.write_html(os.path.join(save_dir, filename))
        fig.show(renderer="browser")
        return fig

    def plot_node_move_progression(self, topology, run_idx, node_idx, save_dir="move_plots",
                                    filename="node_move_progression.html", max_periods=None):
        data = np.load(os.path.join(self.run_dir(topology, run_idx), f"node_{node_idx}", "snapshots.npz"))
        P_snaps, Q_snaps = data["P"], data["Q"]
        n_periods = len(P_snaps) if max_periods is None else min(max_periods, len(P_snaps))

        colorscale = [[0.0, "blue"], [0.25, "blue"], [0.25, "cyan"], [0.5, "cyan"],
                      [0.5, "orange"], [0.75, "orange"], [0.75, "red"], [1.0, "red"]]
        fig = make_subplots(rows=1, cols=n_periods,
                             subplot_titles=[f"gen {i * self.period}" for i in range(n_periods)])

        for i in range(n_periods):
            x_r = P_snaps[i].reshape(-1, self.game.k, self.game.l_t)
            y_r = Q_snaps[i].reshape(-1, self.game.k, self.game.l_t)
            codes = (x_r + 2 * y_r).reshape(-1, self.game.k * self.game.l_t)
            fig.add_trace(go.Heatmap(z=codes, colorscale=colorscale, zmin=-0.5, zmax=3.5,
                                      showscale=(i == n_periods - 1)), row=1, col=i + 1)

        fig.update_layout(title=f"Move Progression - {topology} run {run_idx} node {node_idx}",
                           yaxis_title="Defender/Attacker Pairs", xaxis_title="Resources over Time Steps")
        os.makedirs(save_dir, exist_ok=True)
        fig.write_html(os.path.join(save_dir, filename))
        fig.show(renderer="browser")
        return fig




In [ ]:

if __name__ == "__main__":
    k, l_t = 10, 40
    n = k * l_t
    lam, chi, t, period, n_repeats = 50, 0.3, 2000, 100, 30

    np.random.seed(42)
    c = np.random.gamma(shape=1, scale=200, size=k)
    v = c
    B_D = B_A = 280

    topologies = {
        "ring_10": np.array([[i, (i + 1) % 10] for i in range(10)] + [[(i + 1) % 10, i] for i in range(10)]),
        "star_10": np.array([[0, i] for i in range(1, 10)] + [[i, 0] for i in range(1, 10)]),
        "line_10": np.array([[i, i + 1] for i in range(9)] + [[i + 1, i] for i in range(9)]),
        "complete_10": np.array([[i, j] for i in range(10) for j in range(10) if i != j]),
        "vanilla": np.array([[0,0]])
    }

    jobs = []
    for topo_name, edges in topologies.items():
        def build_algo(edges=edges, topo_name=topo_name):
            return Graph_PDCoEA_Defendit(
                alpha=0, beta=0, chi=chi, l=lam, n=n, t=t, edge_set=edges, num_nodes= 1 if edges.shape[0] == 1 else 10,
                k=k, l_t=l_t, v=v, c=c, B_D=B_D, B_A=B_A,
                timestep=period, topology_name=topo_name,
            )
        jobs.append({"name": topo_name, "build_algo": build_algo, "n_repeats": n_repeats,
                     "compute_champions": True})

    ExperimentQueue(base_dir="overnight_topology_experiment_low_B").run(jobs)

2026-08-16 01:55:59,549 INFO Starting queue with 5 job(s): ['ring_10', 'star_10', 'line_10', 'complete_10', 'vanilla']
2026-08-16 01:55:59,551 INFO [ring_10] starting
2026-08-16 01:55:59,553 INFO   run_0: starting
100%|██████████| 2000/2000 [00:05<00:00, 379.96it/s]
2026-08-16 01:56:10,672 INFO   run_0: done in 11.1s
2026-08-16 01:56:10,673 INFO   run_1: starting
100%|██████████| 2000/2000 [00:03<00:00, 503.32it/s]
2026-08-16 01:56:20,001 INFO   run_1: done in 9.3s
2026-08-16 01:56:20,002 INFO   run_2: starting
100%|██████████| 2000/2000 [00:03<00:00, 511.35it/s]
2026-08-16 01:56:29,186 INFO   run_2: done in 9.2s
2026-08-16 01:56:29,187 INFO   run_3: starting
100%|██████████| 2000/2000 [00:04<00:00, 474.95it/s]
2026-08-16 01:56:38,636 INFO   run_3: done in 9.4s
2026-08-16 01:56:38,637 INFO   run_4: starting
100%|██████████| 2000/2000 [00:03<00:00, 511.06it/s]
2026-08-16 01:56:47,827 INFO   run_4: done in 9.2s
2026-08-16 01:56:47,829 INFO   run_5: starting
100%|██████████| 2000/2000 [00

In [ ]:
import os
import glob
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def group_sort_by_color(codes, n_groups):
    n_rows = codes.shape[0]
    n_groups = min(n_groups, n_rows)

    group_ids = np.arange(n_rows) % n_groups
    group_means = np.array([codes[group_ids == g].mean() for g in range(n_groups)])
    group_order = np.argsort(group_means)

    row_order = np.concatenate([np.where(group_ids == g)[0] for g in group_order])

    return codes[row_order]


def plot_all_move_progressions(root_dir="overnight_topology_experiment_4", run_idx=0, gen_increment=400, save_dir="move_plots_batch", max_gen=None, sort_rows=True, n_groups=20):
    """
    Scans a root directory for all topologies and nodes within a specific run,
    and plots move combinations at specific generation increments.
    """
    if not os.path.exists(root_dir):
        raise ValueError(f"Root directory '{root_dir}' does not exist.")

    topologies = [d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))]

    colorscale = [
        [0.0, "blue"], [0.25, "blue"],
        [0.25, "cyan"], [0.5, "cyan"],
        [0.5, "orange"], [0.75, "orange"],
        [0.75, "red"], [1.0, "red"],
    ]

    for topology in topologies:
        topo_run_dir = os.path.join(root_dir, topology, f"run_{run_idx}")

        if not os.path.exists(topo_run_dir):
            print(f"Skipping {topology}: Data for run_{run_idx} not found at {topo_run_dir}")
            continue

        output_run_dir = os.path.join(save_dir, topology, f"run_{run_idx}")
        os.makedirs(output_run_dir, exist_ok=True)

        params_path = os.path.join(topo_run_dir, "params.npz")
        if not os.path.exists(params_path):
            print(f"Skipping {topology}: params.npz not found.")
            continue

        params = np.load(params_path, allow_pickle=True)
        k = int(params["k"])
        l_t = int(params["l_t"])
        period = int(params["timestep"])

        node_dirs = sorted(glob.glob(os.path.join(topo_run_dir, "node_*")))

        for node_dir in node_dirs:
            node_name = os.path.basename(node_dir)
            snapshot_path = os.path.join(node_dir, "snapshots.npz")

            if not os.path.exists(snapshot_path):
                continue

            data = np.load(snapshot_path)
            P_snaps = data["P"]
            Q_snaps = data["Q"]

            total_snapshots = len(P_snaps)
            max_available_gen = (total_snapshots - 1) * period
            limit_gen = max_available_gen if max_gen is None else min(max_gen, max_available_gen)

            target_indices = []
            for g in range(0, limit_gen + 1, gen_increment):
                idx = int(round(g / period))
                if idx < total_snapshots and idx not in target_indices:
                    target_indices.append(idx)

            last_valid_idx = int(round(limit_gen / period))
            if last_valid_idx < total_snapshots and last_valid_idx not in target_indices:
                target_indices.append(last_valid_idx)

            if not target_indices:
                print(f"No valid generations found to plot for {topology} - {node_name}.")
                continue

            actual_gens = [i * period for i in target_indices]
            n_plots = len(target_indices)

            fig = make_subplots(rows=1, cols=n_plots, subplot_titles=[f"gen {g}" for g in actual_gens])

            for col_idx, snap_idx in enumerate(target_indices):
                x = P_snaps[snap_idx]
                y = Q_snaps[snap_idx]

                x_r = x.reshape(x.shape[0], k, l_t)
                y_r = y.reshape(y.shape[0], k, l_t)

                codes = (x_r + 2 * y_r).reshape(x.shape[0], k * l_t)

                if sort_rows:
                    codes = group_sort_by_color(codes, n_groups)

                fig.add_trace(
                    go.Heatmap(z=codes, colorscale=colorscale, zmin=-0.5, zmax=3.5, showscale=(col_idx == n_plots - 1)),
                    row=1, col=col_idx + 1,
                )

            fig.update_layout(
                title=f"Move Progression - {topology} Run {run_idx} {node_name.capitalize()}" + (" (Sorted)" if sort_rows else ""),
                yaxis_title="Sorted Defender/Attacker Pairs" if sort_rows else "Defender/Attacker Pairs",
                margin=dict(b=80)
            )

            fig.update_yaxes(showticklabels=False)

            mid_col = (n_plots + 1) // 2
            fig.update_xaxes(title_text="Resources over Time Steps", row=1, col=mid_col)

            filename = f"move_progression_{node_name}.html"
            fig.write_html(os.path.join(output_run_dir, filename))
if __name__ == "__main__":
    root_directory = "overnight_topology_experiment_4"
    save_directory = "move_plots_batch_2"

    # Dynamically count the maximum number of runs across any topology
    max_runs = 0
    if os.path.exists(root_directory):
        for topo in os.listdir(root_directory):
            topo_path = os.path.join(root_directory, topo)
            if os.path.isdir(topo_path):
                runs = [d for d in os.listdir(topo_path) if d.startswith("run_")]
                max_runs = max(max_runs, len(runs))

    print(f"Detected up to {max_runs} runs. Processing...")

    # Iterate through each run and plot all topologies found
    for i in range(max_runs):
        print(f"Generating plots for run_{i} across all topologies...")
        plot_all_move_progressions(
            root_dir=root_directory,
            run_idx=i,
            gen_increment=400,
            save_dir=save_directory,
            max_gen=2000
        )

    print(f"All runs processed. Check the '{save_directory}' folder.")
    

Detected up to 30 runs. Processing...
Generating plots for run_0 across all topologies...
Generating plots for run_1 across all topologies...
Generating plots for run_2 across all topologies...
Generating plots for run_3 across all topologies...
Generating plots for run_4 across all topologies...
Generating plots for run_5 across all topologies...
Generating plots for run_6 across all topologies...
Generating plots for run_7 across all topologies...
Generating plots for run_8 across all topologies...
Generating plots for run_9 across all topologies...
Generating plots for run_10 across all topologies...
Generating plots for run_11 across all topologies...
Generating plots for run_12 across all topologies...
Generating plots for run_13 across all topologies...
Generating plots for run_14 across all topologies...
Generating plots for run_15 across all topologies...
Generating plots for run_16 across all topologies...
Generating plots for run_17 across all topologies...
Generating plots f

In [ ]:
class NodeChampionComparison(SnapshotComparison):
    def load_node_snapshots(self, topology, run_idx, node):
        data = np.load(os.path.join(self.run_dir(topology, run_idx), f"node_{node}", "snapshots.npz"))
        return data["P"], data["Q"]

    def compare_node_run(self, topology, run_idx, node):
        P_A, Q_A = self.load_node_snapshots(topology, run_idx, node)
        P_B, Q_B = self.load_node_snapshots(self.topology_B, run_idx, 0)
        n_periods = min(P_A.shape[0], P_B.shape[0])

        def_champ_A, def_champ_B, atk_champ_A, atk_champ_B = [], [], [], []
        perf = {k: [] for k in ("performance_defender_A", "performance_defender_B",
                                 "performance_attacker_A", "performance_attacker_B")}

        for i in range(n_periods):
            X_A, Y_A, X_B, Y_B = P_A[i], Q_A[i], P_B[i], Q_B[i]

            def_champ_A.append(X_A[np.argmax(self.game.payoff_matrix_defender(X_A, Y_B).min(axis=1))])
            def_champ_B.append(X_B[np.argmax(self.game.payoff_matrix_defender(X_B, Y_A).min(axis=1))])
            atk_champ_A.append(Y_A[np.argmax(self.game.payoff_matrix_attacker(X_B, Y_A).min(axis=0))])
            atk_champ_B.append(Y_B[np.argmax(self.game.payoff_matrix_attacker(X_A, Y_B).min(axis=0))])

            V_pool = np.array(atk_champ_A + atk_champ_B)
            U_pool = np.array(def_champ_A + def_champ_B)

            perf["performance_defender_A"].append(self.game.payoff_matrix_defender(X_A, V_pool).min(axis=1).max())
            perf["performance_defender_B"].append(self.game.payoff_matrix_defender(X_B, V_pool).min(axis=1).max())
            perf["performance_attacker_A"].append(self.game.payoff_matrix_attacker(U_pool, Y_A).min(axis=0).max())
            perf["performance_attacker_B"].append(self.game.payoff_matrix_attacker(U_pool, Y_B).min(axis=0).max())

        return {k: np.array(v) for k, v in perf.items()}


def run_node_comparisons(root_dir, topologies, n_repeats, out_dir="node_comparisons", resume=True):
    os.makedirs(out_dir, exist_ok=True)
    logger = logging.getLogger("node_comparison")
    logger.setLevel(logging.INFO)
    logger.handlers = [logging.FileHandler(os.path.join(out_dir, "queue.log")), logging.StreamHandler()]
    for h in logger.handlers:
        h.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))

    for topo in topologies:
        if topo == "vanilla":
            continue
        cmp = NodeChampionComparison(root_dir, topo, "vanilla")

        for run_idx in range(n_repeats):
            meta_path = os.path.join(cmp.run_dir(topo, run_idx), "meta.json")
            if not os.path.exists(meta_path):
                logger.error(f"{topo} run_{run_idx}: missing meta.json, skipping")
                continue
            num_nodes = json.load(open(meta_path))["num_nodes"]

            for node in range(num_nodes):
                save_path = os.path.join(out_dir, topo, f"run_{run_idx}", f"node_{node}.npz")
                done = save_path + ".done"
                if resume and os.path.exists(done):
                    continue
                os.makedirs(os.path.dirname(save_path), exist_ok=True)
                try:
                    perf = cmp.compare_node_run(topo, run_idx, node)
                    np.savez(save_path, **perf)
                    open(done, "w").close()
                    logger.info(f"{topo} run_{run_idx} node_{node}: done")
                except Exception as e:
                    logger.error(f"{topo} run_{run_idx} node_{node}: FAILED: {e}")
                    logger.error(traceback.format_exc())

    logger.info("Node comparison queue finished.")


if __name__ == "__main__":
    topologies = ["ring_10", "star_10", "line_10", "complete_10"]
    run_node_comparisons(root_dir="overnight_topology_experiment_4", topologies=topologies,
                          n_repeats=30, out_dir="node_comparisons")

In [18]:
class NodeComparisonPlotter:
    def __init__(self, comparisons_dir, period=100):
        self.comparisons_dir = comparisons_dir
        self.period = period

    def load_node_runs(self, topology, node, n_runs):
        results = []
        for run_idx in range(n_runs):
            path = os.path.join(self.comparisons_dir, topology, f"run_{run_idx}", f"node_{node}.npz")
            data = np.load(path)
            results.append({k: data[k] for k in data.files})
        return {key: np.stack([r[key] for r in results]) for key in results[0]}

    def discover_topologies(self):
        return sorted(d for d in os.listdir(self.comparisons_dir)
                      if os.path.isdir(os.path.join(self.comparisons_dir, d)))

    def discover_nodes(self, topology, run_idx=0):
        run_dir = os.path.join(self.comparisons_dir, topology, f"run_{run_idx}")
        files = glob.glob(os.path.join(run_dir, "node_*.npz"))
        return sorted(int(os.path.basename(f).split("_")[1].split(".")[0]) for f in files)

    def plot_whisker(self, topology, node, n_runs, metric="performance_defender",
                      save_dir="whisker_plots", filename=None, label_A=None, label_B=None, show=False):
        results = self.load_node_runs(topology, node, n_runs)
        A, B = results[f"{metric}_A"], results[f"{metric}_B"]
        periods = np.arange(A.shape[1]) * self.period
        label_A, label_B = label_A or topology, label_B or "vanilla"
        filename = filename or f"{metric}.html"

        os.makedirs(save_dir, exist_ok=True)
        fig = go.Figure()
        for i in range(A.shape[1]):
            fig.add_trace(go.Box(y=A[:, i], x=[periods[i]] * A.shape[0], name=label_A,
                                  marker_color="red", legendgroup=label_A, showlegend=(i == 0), offsetgroup=label_A))
            fig.add_trace(go.Box(y=B[:, i], x=[periods[i]] * B.shape[0], name=label_B,
                                  marker_color="blue", legendgroup=label_B, showlegend=(i == 0), offsetgroup=label_B))

        fig.update_layout(title=f"{metric} over time - {topology} node {node}",
                           xaxis_title="Time (f_evals)", yaxis_title=metric, boxmode="group")
        fig.write_html(os.path.join(save_dir, filename))
        if show:
            fig.show(renderer="browser")
        return fig

    def plot_all(self, n_runs, save_dir="whisker_plots", metrics=("performance_defender", "performance_attacker")):
        for topology in self.discover_topologies():
            for node in self.discover_nodes(topology):
                node_dir = os.path.join(save_dir, topology, f"node_{node}")
                for metric in metrics:
                    self.plot_whisker(topology=topology, node=node, n_runs=n_runs, metric=metric,
                                       save_dir=node_dir, filename=f"{metric}.html")
                print(f"{topology} node_{node}: done")


if __name__ == "__main__":
    plotter = NodeComparisonPlotter(comparisons_dir="node_comparisons", period=100)
    plotter.plot_all(n_runs=30)

complete_10 node_0: done
complete_10 node_1: done
complete_10 node_2: done
complete_10 node_3: done
complete_10 node_4: done
complete_10 node_5: done
complete_10 node_6: done
complete_10 node_7: done
complete_10 node_8: done
complete_10 node_9: done
line_10 node_0: done
line_10 node_1: done
line_10 node_2: done
line_10 node_3: done
line_10 node_4: done
line_10 node_5: done
line_10 node_6: done
line_10 node_7: done
line_10 node_8: done
line_10 node_9: done
ring_10 node_0: done
ring_10 node_1: done
ring_10 node_2: done
ring_10 node_3: done
ring_10 node_4: done
ring_10 node_5: done
ring_10 node_6: done
ring_10 node_7: done
ring_10 node_8: done
ring_10 node_9: done
star_10 node_0: done
star_10 node_1: done
star_10 node_2: done
star_10 node_3: done
star_10 node_4: done
star_10 node_5: done
star_10 node_6: done
star_10 node_7: done
star_10 node_8: done
star_10 node_9: done
